# 08 · Profile: Census QWI, hiring and new-hire pay by industry, age and education

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/census_qwi.md`.

**Why this source matters.** QCEW (notebook 07) says how many jobs each NYC industry *has*. It doesn't say how many people each industry hires, or what new hires earn. For a graduate, those are the questions: *who is hiring people like me, and what do they pay at the start?* The Census Bureau's Quarterly Workforce Indicators (QWI) answer both by county, industry, age and education. It's the only government source with hires by age for each borough.

**What this notebook does:** streams the two NY files (age, education), keeps the five boroughs, checks grain, status flags and additivity, ties to QCEW, and ends with an issues list and proposed staging table. It changes nothing.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
BOROUGHS = {"36005": "Bronx", "36047": "Brooklyn", "36061": "Manhattan", "36081": "Queens", "36085": "Staten Island"}

man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key == "census_qwi"][["file", "downloaded_utc", "bytes"]]
display(FILES)
print((RAW / FILES.file[FILES.file.str.endswith("version_qwi.txt")].iloc[0]).read_text())

,file,downloaded_utc,bytes
17,hiring/census_qwi/qwi_ny_sa_f_gc_ns_op_u.csv.gz,2026-10-12T17:06:04+00:00,167162957
18,hiring/census_qwi/qwi_ny_se_f_gc_ns_op_u.csv.gz,2026-10-12T17:06:34+00:00,120733081
19,hiring/census_qwi/qwi_ny.md5sum,2026-10-12T17:06:35+00:00,8184
20,hiring/census_qwi/version_qwi.txt,2026-10-12T17:06:36+00:00,197


QWI_F NY 36 2000:1-2025:4 V4.14.0 R2026Q3 qwipu_ny_20260825_1103
QWI_FA NY 36 2000:1-2025:3 V4.14.0 R2026Q3 qwipu_ny_20260825_1103
QWI_FS NY 36 2000:1-2025:3 V4.14.0 R2026Q3 qwipu_ny_20260825_1103



## 0. Source header

| | Census Bureau LEHD Quarterly Workforce Indicators, New York (`census_qwi`) |
|---|---|
| **Publisher** | U.S. Census Bureau, Longitudinal Employer-Household Dynamics (LEHD) program |
| **Files** | `qwi_ny_sa_f_gc_ns_op_u.csv.gz` (sex × age) and `qwi_ny_se_f_gc_ns_op_u.csv.gz` (sex × education). The name encodes the cut: `f` = all firms, `gc` = county, `ns` = NAICS sectors, `op` = private ownership, `u` = not seasonally adjusted ([LEHD file naming](https://lehd.ces.census.gov/data/schema/latest/lehd_public_use_schema.html)) |
| **Release** | R2026Q3 (schema V4.14.0); data 2000Q1 → 2025Q4 (all-worker file) / 2025Q3 (age and sex/education files) |
| **How it was collected** | **Administrative records linked together**: state unemployment-insurance wage records (one row per worker × employer × quarter) + QCEW employer data + Census demographic files. Hires and separations come from following each worker across quarters. Education is not in the wage records: it's imputed from the ACS / 2000 Census, and only for workers aged 25+ |
| **Confidentiality** | Every number is noise-infused ("fuzzed"); small cells are suppressed. Each value has a status flag column `s<Var>` |
| **License** | Public domain |

**Status flags** (from LEHD `label_flags.csv`): `-2` no data in this category, `-1` not computable, `1` OK, `5` suppressed (below publication standard), `9` significantly distorted (fuzzed value released), `6`/`7`/`10`/`11`/`12` derived or aggregate values.

In [2]:
# Indicators we need for the brief questions (hires, separations, jobs, new-hire earnings) + their flags
KEY = ["geo_level", "geography", "ind_level", "industry", "ownercode", "sex", "agegrp", "education", "year", "quarter"]
VARS = ["Emp", "EmpS", "EmpTotal", "HirA", "HirN", "Sep", "EarnS", "EarnHirNS", "FrmJbC"]
USE = KEY + ["race", "ethnicity", "firmage", "firmsize", "periodicity", "seasonadj"] + VARS + ["s" + v for v in VARS]

def stream(path):
    # Read the whole state file in chunks: count everything, keep only the five boroughs
    n, keep, geo = 0, [], {}
    for ch in pd.read_csv(path, dtype=str, keep_default_na=False, usecols=USE, chunksize=500_000):
        n += len(ch)
        for k, v in ch.geo_level.value_counts().items(): geo[k] = geo.get(k, 0) + v
        keep.append(ch[ch.geography.isin(BOROUGHS)])
    out = pd.concat(keep, ignore_index=True)
    print(f"{path.name}: {n:,} rows in the state file | geo levels {geo} | kept {len(out):,} borough rows")
    return out

hdr = pd.read_csv(RAW / FILES.file.iloc[0], dtype=str, nrows=0)
print(f"Columns in the file: {hdr.shape[1]} | first 17 are dimensions: {list(hdr.columns[:17])}")
qa = stream(RAW / FILES.file[FILES.file.str.contains("_sa_")].iloc[0])
qe = stream(RAW / FILES.file[FILES.file.str.contains("_se_")].iloc[0])

Columns in the file: 81 | first 17 are dimensions: ['periodicity', 'seasonadj', 'geo_level', 'geography', 'ind_level', 'industry', 'ownercode', 'sex', 'agegrp', 'race', 'ethnicity', 'education', 'firmage', 'firmsize', 'year', 'quarter', 'agg_level']


qwi_ny_sa_f_gc_ns_op_u.csv.gz: 3,553,254 rows in the state file | geo levels {'C': 3494340, 'S': 58914} | kept 283,041 borough rows


qwi_ny_se_f_gc_ns_op_u.csv.gz: 2,368,836 rows in the state file | geo levels {'C': 2329560, 'S': 39276} | kept 188,694 borough rows


**Load findings**
- 81 columns: 17 dimensions + 32 indicators + 32 status flags. We read 10 dimensions + 9 indicators + their flags.
- The age file has 3.55M rows for all of New York State (62 counties + the state row); the education file 2.37M. The five boroughs are 283,041 and 188,694 rows.
- The all-worker file (`QWI_F`) runs to 2025Q4; the version file says the age and sex/education files stop at 2025Q3, but both contain 2025Q4 rows (see §5 for what's in them).

## 2. What one row is (grain)

**Presumed key:** county × industry × ownership × sex × age (or education) × year × quarter. The other dimensions (race, ethnicity, firm age, firm size) should be constant.

In [3]:
for name, d in [("age file", qa), ("education file", qe)]:
    const = {c: d[c].unique().tolist() for c in ["periodicity", "seasonadj", "ownercode", "race", "ethnicity", "firmage", "firmsize"]}
    print(f"{name}: constants {const}")
    print(f"  duplicates on the key: {d.duplicated(KEY).sum()} | ind_level {d.ind_level.value_counts().to_dict()}")
    print(f"  sex {sorted(d.sex.unique())} | agegrp {sorted(d.agegrp.unique())} | education {sorted(d.education.unique())}")
    print(f"  years {d.year.min()}-{d.year.max()} | latest quarter {d[d.year == d.year.max()].quarter.max()}")

age file: constants {'periodicity': ['Q'], 'seasonadj': ['U'], 'ownercode': ['A05'], 'race': ['A0'], 'ethnicity': ['A0'], 'firmage': ['0'], 'firmsize': ['0']}


  duplicates on the key: 0 | ind_level {'S': 269001, 'A': 14040}


  sex ['0', '1', '2'] | agegrp ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08'] | education ['E0']
  years 2000-2025 | latest quarter 4
education file: constants {'periodicity': ['Q'], 'seasonadj': ['U'], 'ownercode': ['A05'], 'race': ['A0'], 'ethnicity': ['A0'], 'firmage': ['0'], 'firmsize': ['0']}


  duplicates on the key: 0 | ind_level {'S': 179334, 'A': 9360}
  sex ['0', '1', '2'] | agegrp ['A00'] | education ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']
  years 2000-2025 | latest quarter 4


**Grain findings**
- Key `geography + industry + sex + agegrp + education + year + quarter` is unique in both files (`ind_level` follows from `industry`).
- Constant in both files: quarterly, not seasonally adjusted, private only (`A05`), all races, all firm ages and sizes.
- Industry = all industries (`00`) + 20 NAICS sectors. No 3-digit detail in these files (`ns` = sectors).
- The age file has every age group with `education = E0`; the education file has every education group with `agegrp = A00`. Age and education are never crossed, so "hires aged 22–24 with a bachelor's" doesn't exist in QWI.

## 3. Column profile: how much of each indicator is usable (2019+)

In [4]:
FLAG = {"-2": "no data", "-1": "not computable", "1": "OK", "5": "suppressed", "9": "distorted", "6": "derived", "7": "derived*", "10": "aggregate", "11": "agg. suppressed", "12": "agg.*"}
def flags(d, label):
    d = d[d.year >= "2019"]
    t = pd.DataFrame({v: d["s" + v].map(FLAG).value_counts(normalize=True).mul(100).round(1) for v in VARS}).fillna(0)
    print(f"{label}: % of 2019+ rows by status flag"); display(t)
    blank_ok = {v: int(((d[v] == "") & (d["s" + v] == "1")).sum()) for v in VARS}
    print("  blank value but flag OK:", {k: v for k, v in blank_ok.items() if v} or "none")
    nonblank_bad = {v: int(((d[v] != "") & d["s" + v].isin(["-2", "-1", "5"])).sum()) for v in VARS}
    print("  value present but flag says missing/suppressed:", {k: v for k, v in nonblank_bad.items() if v} or "none")
flags(qa, "Age file"); flags(qe, "Education file")

Age file: % of 2019+ rows by status flag


,Emp,EmpS,EmpTotal,HirA,HirN,Sep,EarnS,EarnHirNS,FrmJbC
OK,97.2,93.7,97.3,96.1,95.8,92.7,90.9,86.4,93.5
not computable,0.0,3.6,0.0,0.0,0.0,3.6,3.6,3.6,3.6
suppressed,2.8,2.7,2.7,3.9,4.2,3.6,5.5,10.0,2.8


  blank value but flag OK: none
  value present but flag says missing/suppressed: none
Education file: % of 2019+ rows by status flag


,Emp,EmpS,EmpTotal,HirA,HirN,Sep,EarnS,EarnHirNS,FrmJbC
OK,96.9,93.4,96.9,96.6,96.4,93.1,93.2,89.2,93.3
not computable,0.0,3.6,0.0,0.0,0.0,3.6,3.6,3.6,3.6
suppressed,3.1,3.0,3.1,3.4,3.6,3.2,3.2,7.1,3.1


  blank value but flag OK: none
  value present but flag says missing/suppressed: none


**Profile findings**
- Status flags are honest: no blank value is flagged OK and no value is present where the flag says suppressed.
- Jobs and hires are 96–97% usable; new-hire earnings (`EarnHirNS`) is the weakest at 86–89% (10% suppressed in the age file).
- **"Not computable" = 3.6%** for stable jobs, separations and earnings: these need the *next* quarter's records, so the latest quarter (2025Q4) is always missing them (and `EarnHirNS` for 2025Q4 too). The newest complete quarter is one behind the newest published quarter.
- No `9` (distorted) flags in the borough slice: everything released is flagged OK.

## 4. Codes and the cells that matter for graduates

The groups closest to a new graduate are age 22–24 (`A03`) and 25–34 (`A04`) in the age file, and some college / associate (`E3`) and bachelor's or more (`E4`) in the education file. How usable are new hires (`HirN`) and new-hire earnings (`EarnHirNS`) for those groups, by borough × sector, 2019+?

In [5]:
def usable(d, dim, codes, var):
    x = d[(d.year >= "2019") & (d.sex == "0") & d[dim].isin(codes) & (d.ind_level == "S")]
    ok = x["s" + var].isin(["1", "9"])  # released values (OK or fuzzed-but-released)
    return (ok.groupby([x.geography.map(BOROUGHS), x[dim]]).mean() * 100).round(0).unstack()
for var in ["HirN", "EarnHirNS"]:
    print(f"% of borough x sector x quarter cells with a released {var} value (2019+, both sexes):")
    display(pd.concat([usable(qa, "agegrp", ["A03", "A04"], var), usable(qe, "education", ["E3", "E4"], var)], axis=1))
print("Industry codes:", sorted(qa.industry.unique()))

% of borough x sector x quarter cells with a released HirN value (2019+, both sexes):


,A03,A04,E3,E4
geography,,,,
Bronx,98.0,96.0,97.0,98.0
Brooklyn,96.0,97.0,96.0,97.0
Manhattan,98.0,97.0,96.0,96.0
Queens,96.0,95.0,94.0,94.0
Staten Island,98.0,99.0,99.0,99.0


% of borough x sector x quarter cells with a released EarnHirNS value (2019+, both sexes):


,A03,A04,E3,E4
geography,,,,
Bronx,85.0,87.0,87.0,87.0
Brooklyn,91.0,93.0,93.0,94.0
Manhattan,89.0,93.0,92.0,93.0
Queens,87.0,91.0,91.0,92.0
Staten Island,85.0,85.0,86.0,85.0


Industry codes: ['00', '11', '21', '22', '23', '31-33', '42', '44-45', '48-49', '51', '52', '53', '54', '55', '56', '61', '62', '71', '72', '81', '92']


**Codes findings**
- For the graduate-age groups (22–24, 25–34) and the two education groups we care about, 94–99% of borough × sector × quarter cells have new hires, and 85–94% have new-hire earnings. Gaps are in small sectors in small boroughs. Good enough for borough × sector views; very small cells need a "not published" state.
- **Education codes** (LEHD `label_education.csv`): `E1` less than high school, `E2` high school, **`E3` some college *or* associate, `E4` bachelor's *or advanced*, `E5` not available: workers aged 24 or younger**. Two consequences for this project:
  1. Associate's graduates are lumped with "some college, no degree", and bachelor's with master's and doctorates. QWI can't isolate either CUNY degree.
  2. Education is only for workers 25+. Most new graduates are under 25 and land in `E5`, not in `E3`/`E4`.
- Industry codes use NAICS ranges (`31-33`, `44-45`, `48-49`), unlike QCEW's first-code style (`31`, `44`, `48`).

## 5. Coverage over time (NYC, all ages, all industries)

In [6]:
tot = qa[(qa.sex == "0") & (qa.agegrp == "A00") & (qa.ind_level == "A")].copy()
for v in ["Emp", "HirN", "Sep", "EarnHirNS"]:
    tot[v] = pd.to_numeric(tot[v], errors="coerce")
nyc_q = tot.groupby(["year", "quarter"])[["Emp", "HirN", "Sep"]].sum(min_count=5)
nyc_q["new_hire_rate_pct"] = (nyc_q.HirN / nyc_q.Emp * 100).round(1)
display(nyc_q.loc["2019":].T)

year                    2019                                        2020                                        2021                                        2022                                        2023             \
quarter                    1          2          3          4          1          2          3          4          1          2          3          4          1          2          3          4          1          2   
Emp                3985923.0  4005153.0  4071754.0  4078090.0  4077027.0  3565772.0  3350080.0  3457097.0  3506369.0  3534894.0  3636459.0  3709592.0  3802073.0  3859033.0  3993056.0  4029044.0  4010121.0  4032436.0   
HirN                495167.0   598452.0   614292.0   576856.0   466370.0   303407.0   400418.0   432178.0   379004.0   535662.0   657304.0   682532.0   526883.0   656021.0   662324.0   559861.0   521749.0   574726.0   
Sep                 567419.0   645394.0   712331.0   707641.0  1065748.0   581151.0   519576.0   596311.0   444076.0   541330.0   689791.0   718593.0   550439.0   637001.0   730291.0   699630.0   593583.0   615364.0   
new_hire_rate_pct       12.4       14.9       15.1       14.1       11.4        8.5       12.0       12.5       10.8       15.2       18.1       18.4       13.9       17.0       16.6       13.9       13.0       14.3   

year                                          2024                                        2025                                   
quarter                    3          4          1          2          3          4          1          2          3          4  
Emp                4121928.0  4109143.0  4112312.0  4216960.0  4273076.0  4271560.0  4287588.0  4280004.0  4319925.0  4312272.0  
HirN                555966.0   509256.0   582218.0   548874.0   570722.0   531071.0   476455.0   691067.0   566311.0   546599.0  
Sep                 664414.0   619840.0   569173.0   607252.0   676718.0   642114.0   583955.0   759040.0   690413.0        NaN  
new_hire_rate_pct       13.5       12.4       14.2       13.0       13.4       12.4       11.1       16.1       13.1       12.7

**Coverage findings**
- Continuous 2000Q1–2025Q4 for all five boroughs. NYC private jobs (beginning of quarter) fall from 4.08M (2020Q1) to 3.35M (2020Q3) and pass the pre-pandemic level in 2023.
- NYC private employers make ~470,000–690,000 new hires a quarter, a new-hire rate of 11–18% of jobs. The rate peaked in late 2021 (18%) and has cooled to 11–16% in 2025.
- `Sep` (separations) is blank for 2025Q4 (needs 2026Q1 records), as expected.

## 6. Relationships: do the parts add up to the whole?

Because of noise infusion, they shouldn't add up *exactly*. How close are they?

In [7]:
def additivity(d, dim, all_code, var="Emp"):
    x = d[(d.year >= "2019") & (d.sex == "0") & (d.ind_level == "A")].copy()
    x[var] = pd.to_numeric(x[var], errors="coerce")
    whole = x[x[dim] == all_code].set_index(["geography", "year", "quarter"])[var]
    parts = x[x[dim] != all_code].groupby(["geography", "year", "quarter"])[var].sum()
    r = (parts / whole - 1).mul(100).dropna()
    return r.describe()[["count", "mean", "min", "max"]].round(2)
print("Sum of age groups vs all ages (Emp), % difference:"); print(additivity(qa, "agegrp", "A00").to_dict())
print("Sum of education groups vs all (Emp), % difference:"); print(additivity(qe, "education", "E0").to_dict())
# sectors vs all industries
x = qa[(qa.year >= "2019") & (qa.sex == "0") & (qa.agegrp == "A00")].copy(); x["Emp"] = pd.to_numeric(x.Emp, errors="coerce")
whole = x[x.ind_level == "A"].set_index(["geography", "year", "quarter"]).Emp
parts = x[x.ind_level == "S"].groupby(["geography", "year", "quarter"]).Emp.sum()
print("Sum of sectors vs all industries (Emp), % difference:", ((parts / whole - 1) * 100).dropna().describe()[["count", "mean", "min", "max"]].round(2).to_dict())
# Same cell in both files (all ages = all education) should be identical
a = qa[(qa.agegrp == "A00")].set_index(["geography", "industry", "sex", "year", "quarter"]).Emp
e = qe[(qe.education == "E0")].set_index(["geography", "industry", "sex", "year", "quarter"]).Emp
j = pd.concat([a.rename("age_file"), e.rename("edu_file")], axis=1, join="inner")
print(f"\nAll-ages row (age file) == all-education row (education file): {(j.age_file == j.edu_file).mean():.1%} of {len(j):,} shared cells")

Sum of age groups vs all ages (Emp), % difference:
{'count': 140.0, 'mean': 0.0, 'min': -0.0, 'max': 0.0}
Sum of education groups vs all (Emp), % difference:


{'count': 140.0, 'mean': -0.0, 'min': -0.0, 'max': 0.0}
Sum of sectors vs all industries (Emp), % difference: {'count': 140.0, 'mean': -0.0, 'min': -0.01, 'max': 0.0}



All-ages row (age file) == all-education row (education file): 100.0% of 31,449 shared cells


**Relationship findings**
- Age groups, education groups and sectors all add up to their totals exactly (max difference 0.01%) for 2019+ employment. Noise is applied at the establishment level, before aggregation, so the published parts and totals are consistent.
- The all-ages row of the age file and the all-education row of the education file are the same numbers (100% of 31,449 shared cells). Staging can stack the two files without double-counting as long as the totals are taken once.

## 7. Tie-out: QWI private jobs vs QCEW private jobs (NYC)

QWI is built on the same unemployment-insurance records as QCEW, so annual private employment should be close. QWI `Emp` is beginning-of-quarter jobs; QCEW is the average of monthly counts.

In [8]:
QCEW = RAW / man.loc[man.key == "nysdol_qcew_annual", "file"].iloc[0]
qc = pd.read_csv(QCEW, dtype=str, keep_default_na=False); qc.columns = [c.strip() for c in qc.columns]
qc = qc[(qc.Area == "New York City") & (qc["Area Type"] == "Labor Market Area") & (qc.Ownership == "Private") & (qc.NAICS == "00")]
qcew = qc.set_index("Year")["Average Employment"].astype(int)
qwi = tot.groupby(["year", "quarter"]).Emp.sum().groupby("year").mean().round()
t = pd.DataFrame({"QCEW private": qcew, "QWI Emp (mean of quarters)": qwi}).loc["2019":"2024"]
t["diff_pct"] = ((t.iloc[:, 1] / t.iloc[:, 0] - 1) * 100).round(2)
display(t)

# What do new hires earn? NYC 2024, private, by education (monthly earnings of stable new hires, all sectors)
y = qe[(qe.year == "2024") & (qe.sex == "0") & (qe.ind_level == "A")].copy()
for v in ["HirN", "EarnHirNS"]: y[v] = pd.to_numeric(y[v], errors="coerce")
y["wsum"] = y.HirN * y.EarnHirNS
s = y.groupby("education")[["HirN", "wsum"]].sum()
s["avg_monthly_new_hire_earnings"] = (s.wsum / s.HirN).round()
print("NYC 2024, new hires and their average monthly earnings, by education:"); display(s[["HirN", "avg_monthly_new_hire_earnings"]])

,QCEW private,QWI Emp (mean of quarters),diff_pct
2019,3916345,4035230.0,3.04
2020,3421539,3612494.0,5.58
2021,3506836,3596828.0,2.57
2022,3793900,3920802.0,3.34
2023,3928359,4068407.0,3.57
2024,4063618,4218477.0,3.81


NYC 2024, new hires and their average monthly earnings, by education:


,HirN,avg_monthly_new_hire_earnings
education,,
E0,2232885,5519.0
E1,305375,4442.0
E2,404322,5229.0
E3,487950,5621.0
E4,536457,8209.0
E5,498844,2968.0


**Tie-out findings**
- QWI private jobs are 2.6–5.6% above QCEW private jobs for NYC. Both come from the same UI system, but they count differently: QWI builds jobs from worker wage records (a beginning-of-quarter job = the worker was paid by that employer in both the previous and the current quarter), while QCEW is the employer's own count of people on the payroll on the 12th of each month. The gap is steady at 3–4% except 2020 (5.6%), when people moved in and out of jobs within quarters. Same lesson as CES: don't mix QWI and QCEW levels in one chart.
- What NYC new hires earn, 2024 (average monthly earnings of new hires who stay the full quarter, private, all sectors):

| Education (age 25+) | New hires 2024 | Avg monthly earnings |
|---|---|---|
| Some college or associate (`E3`) | 487,950 | **$5,621** (~$67k a year) |
| Bachelor's or advanced (`E4`) | 536,457 | **$8,209** (~$99k a year) |
| Age 24 or younger, education unknown (`E5`) | 498,844 | $2,968 (~$36k a year, includes part-time and teen jobs) |

These are earnings, not hourly wages: part-time work pulls them down. They line up in rank with the NYS DOL entry wages (nb 03) but are not the same measure.

## 8. Fitness for purpose and 9. Issues list

### 8. Fitness for purpose

| Question | Fit | Why |
|---|---|---|
| Q1: which NYC industries hire the most, by borough | Use | New hires by borough × sector × quarter, 2000–2025, 96%+ released |
| Q1: who gets hired: young workers | yes | Hires by age group (22–24, 25–34) by borough × sector. The best government proxy for "where do people at graduation age get hired" |
| Q4: what new hires earn, by industry | with caveats | Monthly earnings of stable new hires, 86–89% released. Earnings ≠ wage rate (part-time included) |
| Q4: new-hire pay by degree | weak | Education only for 25+, associate lumped with "some college", bachelor's with graduate degrees, and education is imputed |
| Hires by occupation | no | QWI has no occupation. Through the BLS matrix (nb 09) only as an estimate |
| Industry detail below the sector | in these files | LEHD publishes 3/4-digit files (`ns` → `n3`, `n4`) if ever needed |

**Recommendation for v1:** use QWI for hiring by industry and borough, with the 22–24 and 25–34 age groups as the "graduate-age" lens, and new-hire earnings as a second pay measure next to NYS DOL entry wages. Don't use the education split as "associate vs bachelor's".

### 9. Issues list

| # | Issue | Rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Status flags per indicator | all | Keep the value only when flag ∈ {1, 6, 9, 10}; otherwise `NULL`. Keep the flag column for released values that are distorted (`9`) |
| 2 | Latest quarter lacks stable-job measures (`Sep`, `EmpS`, `EarnS`, `EarnHirNS`) | 2025Q4 | Expected; mark `quarter_complete = false` for the latest published quarter |
| 3 | Age and education in separate files, never crossed | both files | Stack into one table with a `breakdown` column (`age` / `education`); totals (`A00`/`E0`) kept once |
| 4 | Education imputed, 25+ only; `E3` = some college or associate; `E4` = bachelor's or more | education file | Label them exactly that way in the dimension; never call `E3` "associate's" |
| 5 | Not seasonally adjusted | all | Compare same quarter year over year, or 4-quarter sums; never quarter-to-quarter |
| 6 | Noise-infused | all | Footnote on every QWI visual; no single-quarter small-cell claims |
| 7 | Levels ≠ QCEW (+2.6–5.6%) | vs QCEW | Separate fact table |
| 8 | NAICS ranges (`31-33`) vs QCEW first codes (`31`) | industry | One industry dimension with both spellings |
| 9 | `sex` breakdown not needed for v1 | 2/3 of rows | Keep `sex = 0` only |

### Proposed staging table

**`stg_census__qwi`**
- **Grain / key:** `(county_fips, naics_sector, breakdown, group_code, year, quarter)`, private ownership, both sexes
- **Columns:** `county_fips`, `naics_sector`, `breakdown` (`age`/`education`/`all`), `group_code` (`A03`, `E4`…), `year`, `quarter`, `emp_begin`, `emp_stable`, `hires_all`, `hires_new`, `separations`, `earn_stable_monthly`, `earn_new_hire_stable_monthly`, `job_creation`, one flag column per measure, `release` (`R2026Q3`)
- **Filters:** 5 boroughs, `sex = 0`, 2019 → latest (row count below)

In [9]:
# Row accounting for the proposed staging filter, recomputed rather than typed
st = pd.concat([qa.assign(breakdown="age"), qe[qe.education != "E0"].assign(breakdown="education")])
st = st[(st.sex == "0") & (st.year >= "2019")]
assert not st.duplicated(["geography", "industry", "agegrp", "education", "year", "quarter"]).any()
print(f"Borough rows in: {len(qa) + len(qe):,} -> staging rows out: {len(st):,} "
      f"(both sexes, 2019Q1-{st.year.max()}Q{st[st.year == st.year.max()].quarter.max()}, totals kept once)")

Borough rows in: 471,735 -> staging rows out: 39,648 (both sexes, 2019Q1-2025Q4, totals kept once)
